# D — Graph-grounded verification (CPU)

The method. Takes a tutor's Bangla answer, breaks it into claims, matches each claim
against the curriculum graph, and returns **supported / contradicted / not-in-curriculum**.

### What this version can and cannot do

Claim matching is lexical — whole-token entity linking plus content-word overlap against
stored facts. No neural model, so it runs in seconds and every verdict is inspectable,
which matters when you have to defend a false positive to a reviewer.

That buys three of the four things a verifier needs:

- **supported** — the claim overlaps a stored fact about an entity it mentions
- **not-in-curriculum** — no entity or no relevant fact; the tutor went outside the syllabus
- **contradicted (numeric)** — the claim states a quantity for an entity whose stored
  `পরিমাণ` fact says something different. This is the case a curriculum verifier catches
  most cleanly, and it is the "tutor says 160 when the book says 140" scenario.

What it **cannot** do is detect open-ended semantic contradiction — a claim that asserts
something contrary to the book without disagreeing on a number or a named entity. That
needs entailment, and it is the honest limitation of a lexical approach. Measure this
version first; only add a model if the numbers show it is the bottleneck.

In [ ]:
import json, glob, re, collections
from pathlib import Path
import pandas as pd

# ---- thresholds ---------------------------------------------------------
OVERLAP_SUPPORT = 0.34   # share of a fact's content words the claim must contain
MIN_CLAIM_WORDS = 3      # shorter fragments are not checkable assertions
# -------------------------------------------------------------------------

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("eval")
WORK.mkdir(parents=True, exist_ok=True)


def find(name, *fallbacks):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for f in fallbacks:
        hits += glob.glob(f)
    if not hits:
        raise SystemExit(f"{name} not found — attach the dataset or notebook output")
    return hits[0]


T = pd.read_csv(find("biology_all_triples.csv",
                     "kg/triples/biology_all_triples.csv",
                     "../kg/triples/biology_all_triples.csv"))
T["subject"] = T.subject.astype(str).str.strip()
T["object"] = T.object.astype(str).str.strip()

A = pd.read_csv(find("tutor_answers.csv",
                     "eval/tutor_answers.csv",
                     "../eval/tutor_answers.csv"))
print(f"{len(T)} triples over {T.subject.nunique()} entities")
print(f"{len(A)} tutor answers from {A.model.nunique()} models")

## 1 — Claims

A claim is a sentence. Bangla sentences end in `।`, and answers also arrive as bullet or
numbered lists, so both are split on. Sentence-level is coarse — a sentence can carry two
assertions — but it needs no model and keeps every verdict traceable to text the reader
can see.

In [ ]:
WORD = re.compile(r"[ঀ-৿]+|[A-Za-z]+|[0-9০-৯]+(?:[.,][0-9০-৯]+)*")
NUMBER = re.compile(r"[0-9০-৯]+(?:[.,][0-9০-৯]+)*")
SENT_SPLIT = re.compile(r"(?<=[।?!])\s+|\n+|(?:^|\s)[-*•]\s+")

BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
SUFFIXES = ["গুলোর", "গুলোকে", "গুলো", "টিকে", "গুলি", "দের", "টির", "টি",
            "য়ের", "এর", "কে", "ের", "রা", "র"]
# Function words carry no evidence; leaving them in makes everything look like a match.
STOP = {"এবং", "বা", "এই", "সেই", "এটি", "এটা", "যে", "যা", "তা", "হয়", "হলো",
        "করে", "থেকে", "জন্য", "সাথে", "মধ্যে", "সব", "অনেক", "কিছু", "করা",
        "হয়ে", "একটি", "একটা", "নয়", "না", "তাই", "কিন্তু", "আর", "ও"}


def lemma(w):
    for s in SUFFIXES:
        if w.endswith(s) and len(w) - len(s) >= 3:
            return w[: -len(s)]
    return w


def toks(s):
    return [lemma(w) for w in WORD.findall(str(s))]


def content(s):
    return {w for w in toks(s) if w not in STOP and len(w) >= 3}


def numbers(s):
    return {n.translate(BN_DIGITS).rstrip(".,") for n in NUMBER.findall(str(s))}


claims = []
for r in A.itertuples():
    for j, sent in enumerate(SENT_SPLIT.split(str(r.answer))):
        sent = sent.strip(" ।\t")
        if len(toks(sent)) < MIN_CLAIM_WORDS:
            continue
        claims.append({"model": r.model, "qid": r.qid, "claim_no": j,
                       "chapter_no": r.chapter_no, "claim": sent})

C = pd.DataFrame(claims)
print(f"{len(C)} claims from {len(A)} answers "
      f"({len(C)/max(len(A),1):.1f} per answer)")
print(C.groupby("model").size().rename("claims").to_string())

## 2 — Link each claim to curriculum entities

Whole-token matching, longest entity first, so `তন্দ্র` cannot match inside `টিস্যুতন্দ্র`.

In [ ]:
GENERIC = {"মাধ্যম", "ধরন", "জিনিস", "সময়", "স্থান", "গুরুত্ব", "নাম",
           "ব্যাপার", "কারণ", "ফলে", "দিক"}

vocab = {}
for e in T.subject.unique():
    tk = tuple(toks(e))
    if tk and len("".join(tk)) >= 4 and e not in GENERIC:
        vocab[tk] = e
by_len = collections.defaultdict(dict)
for tk, e in vocab.items():
    by_len[len(tk)][tk] = e
MAXN = max(by_len)

FACTS = collections.defaultdict(list)
for r in T.itertuples():
    FACTS[r.subject].append((r.relation, r.object, r.triple_id))


def link(text):
    tk, found, covered = toks(text), [], set()
    for n in range(MAXN, 0, -1):
        table = by_len.get(n)
        if not table:
            continue
        for i in range(len(tk) - n + 1):
            if any(j in covered for j in range(i, i + n)):
                continue
            hit = table.get(tuple(tk[i:i + n]))
            if hit:
                found.append(hit)
                covered.update(range(i, i + n))
    return found


C["entities"] = C.claim.map(link)
C["n_entities"] = C.entities.map(len)
print(f"claims linked to >=1 curriculum entity: "
      f"{(C.n_entities > 0).sum()}/{len(C)} ({(C.n_entities > 0).mean()*100:.0f}%)")

## 3 — Verify

For each linked entity the claim is scored against every stored fact about it. The score is
the share of the *fact's* content words present in the claim — asymmetric on purpose, since
a long answer sentence should still count as supporting a short fact.

A numeric disagreement outranks a textual match: if the claim gives a quantity for an
entity whose `পরিমাণ` fact gives a different one, that is a contradiction regardless of how
much other wording lines up.

In [ ]:
def verify(claim, entities):
    cw, cn = content(claim), numbers(claim)
    best = {"verdict": "not_in_curriculum", "score": 0.0,
            "entity": entities[0] if entities else "", "relation": "",
            "evidence": "", "triple_id": ""}
    if not entities:
        return best

    for ent in entities:
        for rel, obj, tid in FACTS.get(ent, []):
            fw = content(obj)
            if not fw:
                continue
            score = len(fw & cw) / len(fw)

            # Numeric disagreement on a stated quantity is the clearest contradiction
            # a curriculum graph can detect, so it takes precedence.
            fn = numbers(obj)
            if rel == "পরিমাণ" and cn and fn and not (cn & fn):
                return {"verdict": "contradicted", "score": 1.0, "entity": ent,
                        "relation": rel, "evidence": obj, "triple_id": tid}

            if score > best["score"]:
                best = {"verdict": "supported" if score >= OVERLAP_SUPPORT
                        else "not_in_curriculum",
                        "score": round(score, 3), "entity": ent, "relation": rel,
                        "evidence": obj, "triple_id": tid}
    return best


V = pd.DataFrame([verify(r.claim, r.entities) for r in C.itertuples()])
V = V.rename(columns={"triple_id": "evidence_triple_id"})
C = pd.concat([C.drop(columns=["entities"]), V], axis=1)

print(C.verdict.value_counts().to_string())
print("\nby model:")
print(pd.crosstab(C.model, C.verdict, normalize="index").mul(100).round(1).to_string())

## 4 — Per-answer view

In [ ]:
per = (C.groupby(["model", "qid"])
       .agg(claims=("claim", "count"),
            supported=("verdict", lambda s: (s == "supported").sum()),
            contradicted=("verdict", lambda s: (s == "contradicted").sum()),
            outside=("verdict", lambda s: (s == "not_in_curriculum").sum()))
       .reset_index())
per["grounded_pct"] = (per.supported / per.claims * 100).round(1)

print("share of an answer's claims found in the curriculum, by model:")
print(per.groupby("model").grounded_pct.describe()[["mean", "50%"]].round(1).to_string())

C.to_csv(WORK / "claim_verdicts.csv", index=False)
per.to_csv(WORK / "answer_verdicts.csv", index=False)
print(f"\nwrote claim_verdicts.csv ({len(C)}) and answer_verdicts.csv ({len(per)})")

## 5 — Sample for manual scoring

These verdicts are the system's output, not ground truth. Precision and recall require
labelling whether each verdict was *right*, which is the same annotation loop as the KG
review — so the sample is written in the format `tools/annotate.html` reads.

Stratified by verdict so the rare classes get enough labels to estimate at all.

In [ ]:
N = 200
parts = []
for v, grp in C.groupby("verdict"):
    take = max(30, round(N * len(grp) / len(C)))
    parts.append(grp.sample(min(take, len(grp)), random_state=11))
S = pd.concat(parts).sample(frac=1, random_state=11).reset_index(drop=True)

S = S.merge(A[["model", "qid", "question"]], on=["model", "qid"], how="left")
S = S.drop(columns=[c for c in ["triple_id"] if c in S.columns])
S.insert(0, "triple_id", [f"V{i:04d}" for i in range(1, len(S) + 1)])
S["subject"] = S.claim
S["relation"] = S.verdict
S["object"] = S.evidence.fillna("")
S["source_text"] = ("প্রশ্ন: " + S.question.fillna("").astype(str)
                    + "   |   matched entity: " + S.entity.fillna("").astype(str)
                    + "  (score " + S.score.astype(str) + ")")

cols = ["triple_id", "chapter_no", "model", "subject", "relation", "object", "source_text"]
S[cols].to_csv(WORK / "verdict_review_sample.csv", index=False)

print(f"{len(S)} verdicts to score -> verdict_review_sample.csv")
print(S.relation.value_counts().to_string())
print("\nIn annotate.html: 1 = the verdict is right, 3 = the verdict is wrong,")
print("2 = right call for the wrong reason (matched the wrong fact).")

In [ ]:
for v in ["supported", "contradicted", "not_in_curriculum"]:
    sub = C[C.verdict == v]
    if not len(sub):
        continue
    print(f"\n{'='*76}\n{v}  ({len(sub)} claims)\n{'='*76}")
    for r in sub.head(3).itertuples():
        print(f"\n  [{r.model}] {str(r.claim)[:110]}")
        if r.entity:
            print(f"     entity   : {r.entity}")
            print(f"     evidence : [{r.relation}] {str(r.evidence)[:80]}  (score {r.score})")